# Step 3 Part E: Whalley-Wilmott no-trade band

Different mechanism from Leland: instead of adjusting volatility, we define a BAND around the true BS delta. As long as our current position stays inside the band, we do nothing (no trade at all). We only trade when the position drifts outside the band -- and even then, we only trade back to the EDGE of the band, not all the way to the exact delta.

The asymptotic band half-width (Whalley & Wilmott, 1997):

$$H_t = \left(\frac{3 \, e^{-rT} \, k \, S^2 \, \Gamma^2}{2 \lambda}\right)^{1/3}$$

where $\Gamma$ is BS gamma, $k$ is the proportional transaction cost rate, $S$ is spot, and $\lambda$ is a risk-aversion parameter (a free parameter of the model -- we'll pick a value and be explicit about it, since this is a standard part of applying this model).

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

train = pd.read_csv("btc_options_train.csv")
train["hour_bucket"] = pd.to_datetime(train["hour_bucket"])
train["sample_date"] = train["hour_bucket"].dt.date

BTC_TRANSACTION_COST_RATE = 0.0005  # 5 bps round-trip, same as Leland
DT_YEARS = 1 / (365 * 24)
RISK_AVERSION_LAMBDA = 60  # free parameter, calibrated: 0.01 produced a band nearly as wide as delta's full range
# (0.39 to 0.92 in delta units), causing WW to trade only 1/24 times -- clearly too wide. Increasing lambda
# (more 'risk-averse') shrinks the band; 60 was chosen to bring the band down to a more plausible fraction of delta.

def bs_delta(S, K, T_years, sigma, option_type, r=0.0):
    if T_years <= 0 or sigma <= 0:
        if option_type == "call":
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T_years) / (sigma * np.sqrt(T_years))
    return norm.cdf(d1) if option_type == "call" else norm.cdf(d1) - 1.0

def bs_gamma(S, K, T_years, sigma, r=0.0):
    if T_years <= 0 or sigma <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T_years) / (sigma * np.sqrt(T_years))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T_years))

def ww_band_half_width(S, gamma, k, lam, r=0.0, T_years=0.01):
    """Whalley-Wilmott asymptotic band half-width, in DELTA units (not price units)."""
    if gamma <= 0:
        return 0.0
    val = (3 * np.exp(-r * T_years) * k * (S ** 2) * (gamma ** 2)) / (2 * lam)
    return val ** (1 / 3)

def load_episode(symbol, sample_date):
    ep = train[(train["symbol"] == symbol) & (train["sample_date"] == sample_date)].sort_values("hour_bucket").reset_index(drop=True)
    ep["T_years"] = ep["time_to_maturity_days"] / 365
    ep["iv_decimal"] = ep["mark_iv"] / 100
    ep["option_mid_usd"] = ep["mid_price"] * ep["underlying_price"]
    ep["option_pnl"] = -ep["option_mid_usd"].diff().fillna(0)
    ep["btc_cost_rate"] = BTC_TRANSACTION_COST_RATE
    ep["bs_delta"] = ep.apply(lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"], row["type"]), axis=1)
    ep["bs_gamma"] = ep.apply(lambda row: bs_gamma(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"]), axis=1)
    ep["band_half_width"] = ep.apply(lambda row: ww_band_half_width(row["underlying_price"], row["bs_gamma"], BTC_TRANSACTION_COST_RATE, RISK_AVERSION_LAMBDA, T_years=row["T_years"]), axis=1)
    return ep

# Reuse the two episodes from the Leland test
otm_episode = load_episode("BTC-10APR20-5000-P", pd.Timestamp("2020-04-01").date())
atm_episode = load_episode("BTC-20NOV20-13750-C", pd.Timestamp("2020-11-01").date())

print("OTM episode band half-widths (delta units):")
print(otm_episode["band_half_width"].describe())
print("\nATM episode band half-widths (delta units):")
print(atm_episode["band_half_width"].describe())

## Sanity check the band width before using it

The band half-width should be small relative to delta itself (e.g. a few percent), not so large it never trades, and not so tiny it's identical to trading every step. Adjust `RISK_AVERSION_LAMBDA` if the numbers above look unreasonable -- larger lambda = smaller band = trades more often; smaller lambda = wider band = trades less often.

## Simulate with the no-trade band: only rebalance to the band edge when outside it

In [ ]:
def simulate_ww_hedge(episode_df, cost_rate_col="btc_cost_rate"):
    n = len(episode_df)
    position = np.zeros(n)
    trade_size = np.zeros(n)
    transaction_cost = np.zeros(n)
    hedge_pnl = np.zeros(n)

    current_position = 0.0
    for i in range(n):
        target_delta = episode_df["bs_delta"].iloc[i]
        band = episode_df["band_half_width"].iloc[i]
        lower, upper = target_delta - band, target_delta + band

        if i == 0:
            # must establish an initial hedge -- go straight to the BS delta
            new_position = target_delta
        elif current_position < lower:
            new_position = lower  # trade back to the near edge, not all the way to target
        elif current_position > upper:
            new_position = upper
        else:
            new_position = current_position  # inside the band -- do nothing

        trade = new_position - current_position
        trade_size[i] = trade

        spot = episode_df["underlying_price"].iloc[i]
        cost_rate = episode_df[cost_rate_col].iloc[i]
        transaction_cost[i] = abs(trade) * spot * (cost_rate / 2)

        if i > 0:
            prev_spot = episode_df["underlying_price"].iloc[i - 1]
            hedge_pnl[i] = current_position * (spot - prev_spot)

        current_position = new_position
        position[i] = current_position

    result = episode_df.copy()
    result["position"] = position
    result["trade_size"] = trade_size
    result["transaction_cost"] = transaction_cost
    result["hedge_pnl"] = hedge_pnl
    result["total_pnl"] = result["option_pnl"] + result["hedge_pnl"] - result["transaction_cost"]
    return result

def simulate_full_hedge(episode_df, target_delta_col, cost_rate_col="btc_cost_rate"):
    n = len(episode_df)
    position = np.zeros(n)
    trade_size = np.zeros(n)
    transaction_cost = np.zeros(n)
    hedge_pnl = np.zeros(n)
    current_position = 0.0
    for i in range(n):
        target = episode_df[target_delta_col].iloc[i]
        trade = target - current_position
        trade_size[i] = trade
        spot = episode_df["underlying_price"].iloc[i]
        cost_rate = episode_df[cost_rate_col].iloc[i]
        transaction_cost[i] = abs(trade) * spot * (cost_rate / 2)
        if i > 0:
            prev_spot = episode_df["underlying_price"].iloc[i - 1]
            hedge_pnl[i] = current_position * (spot - prev_spot)
        current_position = target
        position[i] = current_position
    result = episode_df.copy()
    result["position"] = position
    result["trade_size"] = trade_size
    result["transaction_cost"] = transaction_cost
    result["hedge_pnl"] = hedge_pnl
    result["total_pnl"] = result["option_pnl"] + result["hedge_pnl"] - result["transaction_cost"]
    return result

for name, ep in [("OTM", otm_episode), ("ATM", atm_episode)]:
    sim_bs = simulate_full_hedge(ep, target_delta_col="bs_delta")
    sim_ww = simulate_ww_hedge(ep)
    print(f"=== {name} episode ===")
    print(f"  BS delta:  cost=${sim_bs['transaction_cost'].sum():.4f}, turnover={sim_bs['trade_size'].abs().sum():.6f}")
    print(f"  WW band:   cost=${sim_ww['transaction_cost'].sum():.4f}, turnover={sim_ww['trade_size'].abs().sum():.6f}")
    n_trades_bs = (sim_bs['trade_size'].abs() > 1e-9).sum()
    n_trades_ww = (sim_ww['trade_size'].abs() > 1e-9).sum()
    print(f"  Number of actual trades: BS={n_trades_bs}/24, WW={n_trades_ww}/24\n")

## Visualize: position (with band) over time for the ATM episode

In [ ]:
sim_ww_atm = simulate_ww_hedge(atm_episode)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(atm_episode["hour_bucket"], atm_episode["bs_delta"], label="BS delta (target)", color="gray", linestyle="--")
ax.fill_between(atm_episode["hour_bucket"], atm_episode["bs_delta"] - atm_episode["band_half_width"],
                 atm_episode["bs_delta"] + atm_episode["band_half_width"], alpha=0.2, label="No-trade band")
ax.plot(sim_ww_atm["hour_bucket"], sim_ww_atm["position"], marker="o", color="darkred", label="Actual WW position")
ax.legend()
ax.set_title("Whalley-Wilmott: position stays inside the band, only trades to the edge")
plt.tight_layout()
plt.show()